# **Libraries**

In [ ]:
%run nb_spn_common

# **Functions**

## **Identity functions**

In [ ]:
# --------------------------------------------------------
#  Associate item identity function
# --------------------------------------------------------
def associate_item_identity(
        access_token: str
      , workspace_id: str
      , item_id: str
      , poll_timeout_seconds: int = 120
      , poll_interval_seconds: int = 5
) -> dict:
    """
    Assign the calling identity (SPN, user, or managed identity) as the
    associated identity (default identity) of a Fabric item.

    This replaces the legacy owner-based dependency with the identity of
    the caller making the API request. Currently supported item types:
    Lakehouse and Eventstream.

    The API is asynchronous - a 202 Accepted response includes a Location
    header pointing to the long-running operation. This function polls that
    operation until it reaches Succeeded, Failed, or the timeout expires.

    :param access_token: OAuth2 bearer token from Entra ID.
    :param workspace_id: GUID of the Fabric workspace.
    :param item_id: GUID of the Fabric item to associate.
    :param poll_timeout_seconds: Maximum seconds to wait for the LRO.
    :param poll_interval_seconds: Seconds between polling attempts.
    :return: The final operation response body as a dict.
    """

    endpoint = (
        f"https://api.fabric.microsoft.com/v1"
        f"/workspaces/{workspace_id}"
        f"/items/{item_id}"
        f"/identities/default/assign?beta=true"
    )

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "assignmentType": "Caller"
    }

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    # If synchronous success (unlikely but handle it)
    if response.status_code == 200:
        print(f"Identity associated immediately for item {item_id}.")
        return response.json()

    # 202 Accepted - poll the Location header
    location_url = response.headers.get("Location")
    if not location_url:
        raise RuntimeError(
            "Received 202 but no Location header for LRO polling."
        )

    print(f"Identity assignment started. Polling LRO for item {item_id}...")

    deadline = time.time() + poll_timeout_seconds

    while time.time() < deadline:
        poll_response = requests.get(location_url, headers={
            "Authorization": f"Bearer {access_token}"
        })
        poll_response.raise_for_status()

        poll_data = poll_response.json()
        status = poll_data.get("status")
        print(f"LRO status: {status}")

        if status == "Succeeded":
            print(f"Identity successfully associated for item {item_id}.")
            return poll_data

        if status in {"Failed", "Error", "Canceled"}:
            raise RuntimeError(
                f"Identity assignment failed with status: {status}. "
                f"Response: {poll_data}"
            )

        time.sleep(poll_interval_seconds)

    raise TimeoutError(
        f"Timed out after {poll_timeout_seconds}s waiting for identity "
        f"assignment on item {item_id}."
    )

# --------------------------------------------------------
#  Get item identity function
# --------------------------------------------------------
def get_item_identity(access_token: str, workspace_id: str, item_id: str) -> dict:
    """
    Retrieve the default identity information for a Fabric item.

    :param access_token: OAuth2 bearer token from Entra ID.
    :param workspace_id: GUID of the Fabric workspace.
    :param item_id: GUID of the Fabric item.
    :return: The item response dict including defaultIdentity info.
    """

    endpoint = (
        f"https://api.fabric.microsoft.com/v1"
        f"/workspaces/{workspace_id}"
        f"/items/{item_id}?include=defaultIdentity"
    )

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()

    data = response.json()

    identity_info = data.get("defaultIdentity", {})
    print(f"Item: {data.get('displayName')} ({item_id})")
    print(f"Identity: {identity_info}")

    return data

# **Operation**

### **Warehouse takeover via Power BI API - required until associated identities supports Warehouse item type.**

In [ ]:
# --------------------------------------------------------
#  Get Warehouse Id
# --------------------------------------------------------
item_name = "<WarehouseName>"
item_type = "Warehouse"
item_id = get_item_id(access_token, item_name, item_type)
print(f"{item_name}: {item_id}")

warehouse_name = item_name
warehouse_id = item_id

# Get token
app = msal.ConfidentialClientApplication(
    client_id
    , authority=f"https://login.microsoftonline.com/{tenant_id}"
    , client_credential=client_secret
)
token = app.acquire_token_for_client(scopes=["https://analysis.windows.net/powerbi/api/.default"])["access_token"]

# Takeover
url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datawarehouses/{warehouse_id}/takeover"
response = requests.post(url, headers={"Authorization": f"Bearer {token}"})

if response.status_code == 200:
    print(f"{warehouse_name} ownership successfully taken over.")
else:
    print(f"Failed: {response.status_code} - {response.text}")

### **Associate item identity**

In [ ]:
# --------------------------------------------------------
#  Associate SPN identity with a Lakehouse
# --------------------------------------------------------
item_name = "<LakehouseName>"
item_type = "Lakehouse"
item_id = get_item_id(access_token, item_name, item_type)
print(f"{item_name}: {item_id}")

associate_item_identity(access_token, workspace_id, item_id)

# --------------------------------------------------------
#  Verify the associated identity
# --------------------------------------------------------
get_item_identity(access_token, workspace_id, item_id)